# Rategoan — 1 Klik: Training 100M + 200M + Auto-Release (GPU T4 gratis, Colab)

Varian **satu-klik** dari `colab-train-gpu.ipynb`: pipeline yang sama persis
(unduh ulang korpus jilid 1-5 dari GitHub Release secara deterministik →
tokenisasi → training) tapi sudah di-preset supaya sekali `Runtime > Run all`
langsung melatih **100M lalu 200M** berurutan dan **otomatis menyimpan**
hasilnya — 100M ke commit git (branch training), 200M ke GitHub Release
(checkpoint ini >100MB, tidak boleh masuk git per kebijakan repo) — tanpa
perlu centang/ubah parameter apa pun secara manual di tengah jalan.

## Syarat SEKALI SAJA sebelum benar-benar "satu klik"

GitHub tetap butuh kredensial untuk menyimpan hasil training — ini tidak
bisa dilewati oleh notebook mana pun (bukan keterbatasan Rategoan, tapi
keamanan GitHub). Supaya run berikutnya benar-benar tinggal 1 klik:

1. Buat GitHub Personal Access Token (scope `repo`) sekali di
   [github.com/settings/tokens](https://github.com/settings/tokens).
2. Di Colab: ikon kunci 🔑 di sidebar kiri → **Add new secret** → nama
   `GITHUB_TOKEN`, isi tokennya → aktifkan akses notebook ini.
3. Selesai. Sel 1 di bawah akan otomatis membaca secret ini setiap run —
   tidak perlu paste token lagi.

Setelah secret tersimpan: buka notebook ini di Colab, pastikan
**Runtime > Change runtime type > T4 GPU**, lalu **Runtime > Run all**.
Selesai — sisanya otomatis (termasuk auto-resume kalau sesi Colab mati
di tengah jalan: tinggal Run all lagi, checkpoint terakhir dimuat balik).

## Ke mana hasilnya tersimpan (namanya persis ini)

| Model | Ukuran file | Tersimpan sebagai | Lokasi |
|---|---|---|---|
| 100M | ≈ 78 MB (di bawah batas 100MB GitHub) | commit git biasa | branch `colab-gpu-training` — `raget/raget-data/neural/raget-neural-massive100m.safetensors` |
| 200M | ≈ 163 MB (di atas batas 100MB GitHub) | GitHub Release asset | tag Release **`checkpoint-200m`** — asset `raget-neural-massive200m.safetensors` |

Laporan training tiap model (perplexity per ronde, dst) ikut ter-commit di
`raget/raget-devlog/neural/training-report-massive100m-round8-colab-gpu.json`
dan `training-report-massive200m-round8-colab-gpu.json` (nama file memuat
"round8" apa adanya — nama internal skrip training, bukan penomoran ulang).

## Catatan jujur

- "Scraping"-nya adalah regenerasi korpus DETERMINISTIK dari data yang
  sudah diarsipkan di GitHub Release milik repo ini (Wikipedia ID, berita,
  Wikibooks, Wikisource, Wiktionary) — bukan crawling situs baru dari nol.
  Ini sengaja: hasilnya reproducible dan tidak melanggar rate-limit sumber.
- Total waktu 1 sesi = anggaran 100M + anggaran 200M (default 300 menit
  masing-masing, bisa diubah di sel training). Kalau melebihi batas sesi
  gratis Colab, tinggal buka lagi & Run all — auto-resume dari checkpoint
  terakhir, nol progres hilang.
- Model boleh bisa dilatih (loss/perplexity turun terukur), tapi ini TIDAK
  otomatis berarti generasinya sudah gramatikal/koheren — itu diukur
  terpisah, lihat `PRD.md` §1 untuk status generasi tiap checkpoint.

In [ ]:
# @title 0. Cek GPU tersedia
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
import torch
print('torch:', torch.__version__)
print('CUDA tersedia:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('PERINGATAN: tidak ada GPU terdeteksi - pastikan Runtime > Change runtime type > T4 GPU sudah dipilih (langkah manual 1).')

In [ ]:
# @title 1. Ambil GitHub token (Secret Colab atau input manual)
import os
GITHUB_TOKEN = None
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = None
if not GITHUB_TOKEN:
    import getpass
    GITHUB_TOKEN = getpass.getpass('Secret GITHUB_TOKEN tidak ditemukan - paste GitHub PAT (scope repo) di sini: ')
assert GITHUB_TOKEN, 'Token GitHub wajib diisi untuk clone repo private + unduh aset Release + push hasil.'
os.environ['GITHUB_TOKEN'] = GITHUB_TOKEN
print('Token diterima (', len(GITHUB_TOKEN), 'karakter).')

In [ ]:
# @title 2. Clone repo Rategoan
REPO_OWNER = 'maetalizer-png'
REPO_NAME = 'Rategoan'
BRANCH = 'main'  # @param {type:"string"}

clone_url = 'https://' + GITHUB_TOKEN + '@github.com/' + REPO_OWNER + '/' + REPO_NAME + '.git'
!rm -rf /content/repo
!git clone --branch {BRANCH} --single-branch {clone_url} /content/repo
%cd /content/repo
!git config user.email 'colab-gpu-runner@rategoan.local'
!git config user.name 'Rategoan Colab GPU Runner'
!ls raget/raget-data/neural/
!ls raget/raget-tools/train-massive50m-colab-gpu.py

In [ ]:
# @title 3. Unduh dump Wikipedia (Release 'Korpus' kalau ada, atau langsung dari Wikimedia)
import requests, os

ASSET_FILENAME = 'idwiki-latest-pages-articles.xml.bz2'
TAG = 'J1'
dump_path = '/content/idwiki-latest-pages-articles.xml.bz2'

def unduh_dari_release():
    meta_headers = {'Authorization': 'Bearer ' + GITHUB_TOKEN, 'Accept': 'application/vnd.github+json'}
    r = requests.get('https://api.github.com/repos/' + REPO_OWNER + '/' + REPO_NAME + '/releases/tags/' + TAG, headers=meta_headers)
    if r.status_code == 404:
        return False
    r.raise_for_status()
    release = r.json()
    asset = next((a for a in release['assets'] if a['name'] == ASSET_FILENAME), None)
    if not asset:
        return False
    asset_id = asset['id']
    print('Aset ditemukan di Release:', TAG, '- id=', asset_id, 'ukuran=', asset['size'], 'bytes')
    dl_headers = {'Authorization': 'Bearer ' + GITHUB_TOKEN, 'Accept': 'application/octet-stream'}
    with requests.get('https://api.github.com/repos/' + REPO_OWNER + '/' + REPO_NAME + '/releases/assets/' + str(asset_id), headers=dl_headers, stream=True, allow_redirects=True) as resp:
        resp.raise_for_status()
        with open(dump_path, 'wb') as f:
            for chunk in resp.iter_content(chunk_size=8 * 1024 * 1024):
                f.write(chunk)
    print('Selesai unduh dari Release:', os.path.getsize(dump_path), 'bytes (harus mendekati', asset['size'], ')')
    return True

def unduh_langsung_wikimedia():
    # Kebijakan Gudang Besar Round 9: raw dump jilid 1 tidak lagi diarsipkan
    # permanen di Release (rak korpus = barang bersih saja). Colab punya akses
    # internet penuh (beda dari sandbox Claude Code Remote yang memblokir
    # dumps.wikimedia.org), jadi unduh langsung dari sumber resmi di sini.
    url = 'https://dumps.wikimedia.org/idwiki/latest/idwiki-latest-pages-articles.xml.bz2'
    print('Release J1 tidak tersedia/asset tidak ditemukan - unduh langsung dari', url)
    with requests.get(url, stream=True) as resp:
        resp.raise_for_status()
        with open(dump_path, 'wb') as f:
            for chunk in resp.iter_content(chunk_size=8 * 1024 * 1024):
                f.write(chunk)
    print('Selesai unduh langsung dari Wikimedia:', os.path.getsize(dump_path), 'bytes')

if not unduh_dari_release():
    unduh_langsung_wikimedia()


In [ ]:
# @title 4. Ekstrak tokenizer (merges+vocab) LANGSUNG dari metadata checkpoint
# Token ID WAJIB identik dengan yang dipakai waktu checkpoint ini dilatih -
# bpe-tokenizer.json terpisah TIDAK dikomit ke git (lihat wikipedia-korpus-manifest.json),
# tapi merges+vocabEntries-nya tersimpan ulang di metadata checkpoint sendiri.
!python3 raget/raget-tools/extract-tokenizer-from-checkpoint.py raget/raget-data/neural/raget-neural-massive50m.safetensors /content/bpe-tokenizer.json
!ls -la /content/bpe-tokenizer.json

In [ ]:
# @title 5. Extract + bersihkan wikitext (regenerasi korpus - deterministik, sama seperti sandbox)
STOP_AT_WORDS = 150000000  # @param {type:"integer"}
!python3 raget/raget-tools/extract-clean-wikipedia.py /content/idwiki-latest-pages-articles.xml.bz2 /content/idwiki-clean.jsonl {STOP_AT_WORDS}

In [ ]:
# @title 6. Tokenisasi + chunking (window 512) memakai tokenizer checkpoint
!python3 raget/raget-tools/tokenize-chunk-corpus.py /content/idwiki-clean.jsonl /content/bpe-tokenizer.json /content/idwiki-chunks.txt 512

In [ ]:
# @title 7. Rechunk ke window 126 (sama seperti round 3/4/5 CPU - attention lebih murah)
!python3 raget/raget-tools/rechunk-corpus-window.py /content/idwiki-chunks.txt /content/idwiki-chunks-126.txt 126

In [ ]:
# @title 8. Unduh 4 aset korpus JILID 2 dari GitHub Release 'Sft'
JILID2_TAG = 'Sft'
JILID2_FILES = ['newspapers-json.tgz', 'train-00000-of-00140.parquet', 'train-00001-of-00140.parquet', 'train-00002-of-00140.parquet']

r2 = requests.get('https://api.github.com/repos/' + REPO_OWNER + '/' + REPO_NAME + '/releases/tags/' + JILID2_TAG, headers=meta_headers)
r2.raise_for_status()
release2 = r2.json()
assets_by_name = {a['name']: a for a in release2['assets']}

os.makedirs('/content/jilid2', exist_ok=True)
for fname in JILID2_FILES:
    asset = assets_by_name[fname]
    out_path = '/content/jilid2/' + fname
    print('Unduh', fname, '(', asset['size'], 'bytes)...')
    with requests.get('https://api.github.com/repos/' + REPO_OWNER + '/' + REPO_NAME + '/releases/assets/' + str(asset['id']), headers=dl_headers, stream=True, allow_redirects=True) as resp:
        resp.raise_for_status()
        with open(out_path, 'wb') as f:
            for chunk in resp.iter_content(chunk_size=8 * 1024 * 1024):
                f.write(chunk)
    actual = os.path.getsize(out_path)
    print('  selesai:', actual, 'bytes (harus =', asset['size'], ') ->', 'OK' if actual == asset['size'] else 'UKURAN TIDAK COCOK!')

In [ ]:
# @title 9. Parse jilid 2 (tgz + 3 parquet) jadi JSONL mentah
!python3 raget/raget-tools/parse-korpus-jilid2.py /content/jilid2/newspapers-json.tgz /content/jilid2/train-00000-of-00140.parquet /content/jilid2/train-00001-of-00140.parquet /content/jilid2/train-00002-of-00140.parquet /content/jilid2

In [ ]:
# @title 10. Bersihkan + dedupe jilid 2 (exact + near MinHash)
!python3 raget/raget-tools/clean-dedupe-korpus-jilid2.py /content/jilid2 /content/jilid2/korpus-jilid2-clean.jsonl /content/jilid2/fase3-clean-report.json

In [ ]:
# @title 8b. Unduh mentah JILID 3/4/5 (Wikibooks tag U, Wikisource+Voyage+Quote tag J34, Wiktionary tag Hhh)
JILID345_ASSETS = [
    ('U', 'idwikibooks-latest-pages-articles.xml.bz2', '/content/idwikibooks-latest-pages-articles.xml.bz2'),
    ('J34', 'idwikivoyage-latest-pages-articles.xml.bz2', '/content/idwikivoyage-latest-pages-articles.xml.bz2'),
    ('J34', 'idwikisource-latest-pages-articles.xml.bz2', '/content/idwikisource-latest-pages-articles.xml.bz2'),
    ('J34', 'idwikiquote-latest-pages-articles.xml.bz2', '/content/idwikiquote-latest-pages-articles.xml.bz2'),
    ('Hhh', 'idwiktionary-latest-pages-articles.xml.bz2', '/content/idwiktionary-latest-pages-articles.xml.bz2'),
]
_release_cache = {}

def unduh_asset(tag, fname, out_path):
    if tag not in _release_cache:
        r = requests.get('https://api.github.com/repos/' + REPO_OWNER + '/' + REPO_NAME + '/releases/tags/' + tag, headers=meta_headers)
        r.raise_for_status()
        _release_cache[tag] = r.json()
    assets_by_name = {a['name']: a for a in _release_cache[tag]['assets']}
    asset = assets_by_name[fname]
    print('Unduh', tag, '/', fname, '(', asset['size'], 'bytes)...')
    with requests.get('https://api.github.com/repos/' + REPO_OWNER + '/' + REPO_NAME + '/releases/assets/' + str(asset['id']), headers=dl_headers, stream=True, allow_redirects=True) as resp:
        resp.raise_for_status()
        with open(out_path, 'wb') as f:
            for chunk in resp.iter_content(chunk_size=8 * 1024 * 1024):
                f.write(chunk)
    actual = os.path.getsize(out_path)
    print('  selesai:', actual, 'bytes (harus =', asset['size'], ') ->', 'OK' if actual == asset['size'] else 'UKURAN TIDAK COCOK!')

for tag, fname, out_path in JILID345_ASSETS:
    unduh_asset(tag, fname, out_path)


In [ ]:
# @title 8c. Bersihkan JILID 3/4/5 (Wikibooks/Wikisource+Voyage+Quote pakai extractor artikel biasa; Wiktionary pakai extractor definisi khusus)
!python3 raget/raget-tools/extract-clean-wikipedia.py /content/idwikibooks-latest-pages-articles.xml.bz2 /content/korpus-jilid3-clean.jsonl 50000000
!python3 raget/raget-tools/extract-clean-wikipedia.py /content/idwikivoyage-latest-pages-articles.xml.bz2 /content/jilid4-wikivoyage.jsonl 50000000
!python3 raget/raget-tools/extract-clean-wikipedia.py /content/idwikisource-latest-pages-articles.xml.bz2 /content/jilid4-wikisource.jsonl 50000000
!python3 raget/raget-tools/extract-clean-wikipedia.py /content/idwikiquote-latest-pages-articles.xml.bz2 /content/jilid4-wikiquote.jsonl 50000000
!python3 raget/raget-tools/extract-clean-wiktionary.py /content/idwiktionary-latest-pages-articles.xml.bz2 /content/korpus-jilid5-clean.jsonl

import json
with open('/content/korpus-jilid4-clean.jsonl', 'w', encoding='utf-8') as out:
    for fname, source in [('jilid4-wikivoyage.jsonl', 'wikivoyage'), ('jilid4-wikisource.jsonl', 'wikisource'), ('jilid4-wikiquote.jsonl', 'wikiquote')]:
        with open('/content/' + fname, encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                obj = json.loads(line)
                obj['source'] = source
                out.write(json.dumps(obj, ensure_ascii=False) + '\n')
print('korpus-jilid4-clean.jsonl (gabungan 3 sumber) ditulis.')


In [ ]:
# @title 10b. (Opsional) Arsipkan korpus BERSIH ke Release - kebijakan Gudang Besar
# Rak korpus = barang bersih saja. Jalankan sel ini kalau mau mengarsipkan
# idwiki-clean.jsonl (jilid 1) dan korpus-jilid2-clean.jsonl (jilid 2) sebagai
# Release asset (tag korpus-jilid-1-clean / korpus-jilid-2-clean) - PAT Colab
# (scope repo) punya izin penuh, beda dari sandbox Claude Code Remote yang
# diblokir dari operasi Release.
ARSIPKAN_KORPUS_BERSIH = False  # @param {type:"boolean"}
if ARSIPKAN_KORPUS_BERSIH:
    import os
    if os.path.exists('/content/idwiki-clean.jsonl'):
        !python3 raget/raget-tools/publish-checkpoint-release.py /content/idwiki-clean.jsonl korpus-jilid-1-clean "Korpus jilid 1 (Wikipedia ID) - bersih" "Hasil extract-clean-wikipedia.py, sebelum tokenisasi."
    if os.path.exists('/content/jilid2/korpus-jilid2-clean.jsonl'):
        !python3 raget/raget-tools/publish-checkpoint-release.py /content/jilid2/korpus-jilid2-clean.jsonl korpus-jilid-2-clean "Korpus jilid 2 (newspapers+HPLT) - bersih" "Hasil parse-korpus-jilid2.py + clean-dedupe-korpus-jilid2.py, sebelum tokenisasi."
    if os.path.exists('/content/korpus-jilid3-clean.jsonl'):
        !python3 raget/raget-tools/publish-checkpoint-release.py /content/korpus-jilid3-clean.jsonl korpus-jilid-3-clean "Korpus jilid 3 (Wikibooks) - bersih" "Hasil extract-clean-wikipedia.py pada idwikibooks."
    if os.path.exists('/content/korpus-jilid4-clean.jsonl'):
        !python3 raget/raget-tools/publish-checkpoint-release.py /content/korpus-jilid4-clean.jsonl korpus-jilid-4-clean "Korpus jilid 4 (Wikisource+Wikivoyage+Wikiquote) - bersih" "Gabungan 3 sumber, field source per baris."
    if os.path.exists('/content/korpus-jilid5-clean.jsonl'):
        !python3 raget/raget-tools/publish-checkpoint-release.py /content/korpus-jilid5-clean.jsonl korpus-jilid-5-clean "Korpus jilid 5 (Wiktionary) - bersih" "Hasil extract-clean-wiktionary.py, hanya definisi bahasa Indonesia."
    print('Setelah ini terkonfirmasi berhasil, raw asset di tag Corpus (newspapers-json.tgz + 3 parquet) boleh dihapus manual dari GitHub - clean archive sudah aman.')
else:
    print('Dilewati (ARSIPKAN_KORPUS_BERSIH=False). Centang param di atas untuk mengarsipkan.')


In [ ]:
# @title 11. Tokenisasi jilid 2/3/4/5 + rechunk window 126 + gabung SEMUA jilid (1-5)
!python3 raget/raget-tools/tokenize-chunk-corpus.py /content/jilid2/korpus-jilid2-clean.jsonl /content/bpe-tokenizer.json /content/jilid2/jilid2-chunks-512.txt 512
!python3 raget/raget-tools/rechunk-corpus-window.py /content/jilid2/jilid2-chunks-512.txt /content/jilid2/jilid2-chunks-126.txt 126

!python3 raget/raget-tools/tokenize-chunk-corpus.py /content/korpus-jilid3-clean.jsonl /content/bpe-tokenizer.json /content/jilid3-chunks-512.txt 512
!python3 raget/raget-tools/rechunk-corpus-window.py /content/jilid3-chunks-512.txt /content/jilid3-chunks-126.txt 126

!python3 raget/raget-tools/tokenize-chunk-corpus.py /content/korpus-jilid4-clean.jsonl /content/bpe-tokenizer.json /content/jilid4-chunks-512.txt 512
!python3 raget/raget-tools/rechunk-corpus-window.py /content/jilid4-chunks-512.txt /content/jilid4-chunks-126.txt 126

!python3 raget/raget-tools/tokenize-chunk-corpus.py /content/korpus-jilid5-clean.jsonl /content/bpe-tokenizer.json /content/jilid5-chunks-512.txt 512
!python3 raget/raget-tools/rechunk-corpus-window.py /content/jilid5-chunks-512.txt /content/jilid5-chunks-126.txt 126

!cat /content/idwiki-chunks-126.txt /content/jilid2/jilid2-chunks-126.txt /content/jilid3-chunks-126.txt /content/jilid4-chunks-126.txt /content/jilid5-chunks-126.txt > /content/korpus-gabungan-126.txt
!wc -l /content/korpus-gabungan-126.txt


In [ ]:
# @title 12. TRAINING GPU - MODE TUNTAS, antrean 50m/100m/200m, auto-resume+auto-commit
MODEL_QUEUE = ['100m', '200m']  # @param {type:"raw"}
BUDGET_MINUTES_PER_MODEL = 300  # @param {type:"number"}
BATCH_SIZE = 96  # @param {type:"number"}
TARGET_PERPLEXITY = 0  # @param {type:"number"}
COMMIT_EVERY_MINUTES = 15  # @param {type:"number"}
PUSH_BRANCH = 'colab-gpu-training'  # @param {type:"string"}
CORPUS_FILE = '/content/korpus-gabungan-126.txt'  # gabungan jilid 1 + jilid 2

for model_size in MODEL_QUEUE:
    print('\n\n========== TRAINING', model_size, '==========')
    !python3 raget/raget-tools/train-massive-colab-gpu.py {model_size} {BUDGET_MINUTES_PER_MODEL} {BATCH_SIZE} {CORPUS_FILE} {TARGET_PERPLEXITY} {COMMIT_EVERY_MINUTES} {PUSH_BRANCH}

print('\nAntrean selesai:', MODEL_QUEUE)
print('Sesi mati di tengah jalan? Klik Run all lagi - checkpoint terakhir (auto-commit tiap', COMMIT_EVERY_MINUTES, 'menit) dimuat balik otomatis, training lanjut, nol progres hilang.')

In [ ]:
# @title 13b. Publikasikan checkpoint >100MB ke Release (OTOMATIS - 1 klik)
# Kebijakan Gudang Besar: checkpoint >100MB (200m ke atas) TIDAK boleh
# di-git-add (sudah otomatis dicegah git_auto_commit()) - harus diunggah
# sebagai Release asset dari sini (PAT Colab, bukan sandbox).
# Default True di notebook 1-klik ini - tidak perlu centang manual.
ARSIPKAN_CHECKPOINT_BESAR = True  # @param {type:"boolean"}
if ARSIPKAN_CHECKPOINT_BESAR:
    import os
    for size in ['200m', '300m', '500m']:
        p = 'raget/raget-data/neural/raget-neural-massive{}.safetensors'.format(size)
        if os.path.exists(p) and os.path.getsize(p) > 95 * 1024 * 1024:
            !python3 raget/raget-tools/publish-checkpoint-release.py {p} checkpoint-{size}
else:
    print('Dilewati (ARSIPKAN_CHECKPOINT_BESAR=False). Centang param di atas untuk mempublikasikan.')

In [ ]:
# @title 13. Commit + push final (jaring pengaman - skrip training sudah auto-commit tiap interval)
for model_size in MODEL_QUEUE:
    ckpt = 'raget/raget-data/neural/raget-neural-massive{}.safetensors'.format(model_size)
    report = 'raget/raget-devlog/neural/training-report-massive{}-round8-colab-gpu.json'.format(model_size)
    !git add {ckpt} {report}
!git commit -m "Round 8: sesi Colab selesai - queue {MODEL_QUEUE}"
!git push origin HEAD:{PUSH_BRANCH}
print('Selesai. Semua checkpoint di MODEL_QUEUE sudah di-push ke branch:', PUSH_BRANCH)
print('Merge branch ini ke main lewat Pull Request di GitHub, atau minta sesi Rategoan berikutnya untuk merge otomatis.')

In [ ]:
# @title 14. Ringkasan akhir - nama file & lokasi persis hasil training
import os
print('=== SELESAI - ringkasan lokasi hasil ===')
for size in MODEL_QUEUE:
    ckpt = 'raget/raget-data/neural/raget-neural-massive{}.safetensors'.format(size)
    report = 'raget/raget-devlog/neural/training-report-massive{}-round8-colab-gpu.json'.format(size)
    ada = os.path.exists(ckpt)
    mb = round(os.path.getsize(ckpt) / 1024 / 1024, 1) if ada else 0
    lokasi = 'GitHub Release tag checkpoint-{}'.format(size) if mb > 95 else 'commit git branch {}'.format(PUSH_BRANCH)
    print('-', size, ':', ckpt, '(' + str(mb) + ' MB)', '->', lokasi if ada else 'BELUM ADA - training belum selesai/gagal')
    print('  laporan:', report, '(ada)' if os.path.exists(report) else '(belum ada)')


## Catatan jujur

- Sel 5-7 dan 9-11 meregenerasi korpus jilid 1 (Wikipedia) dan jilid 2 (berita+web) dari Release GitHub secara DETERMINISTIK (skrip identik dengan yang dipakai di sandbox), bukan mengunduh korpus siap-pakai - satu-satunya cara membawa data besar ke Colab tanpa upload manual berulang.
- Total korpus gabungan saat ini (jilid 1 + jilid 2) TERUKUR ~657,7 juta token - lihat `raget-data/jsonl/external/korpus-jilid2-manifest.json` untuk rincian per sumber. BELUM 1 miliar (Release jilid 2 baru memuat 3 dari 140 shard sumber) - menambah jilid 3 dst tinggal jalankan pipeline `raget-tools/parse-korpus-jilid2.py` + `clean-dedupe-korpus-jilid2.py` dengan tag Release baru (lihat `docs/tambah-data-baru.md`).
- Token/detik GPU T4 sungguhan HANYA diketahui setelah sel 12 selesai jalan - lihat `training-report-massive{50m,100m,200m}-round8-colab-gpu.json` untuk angka nyata, bukan estimasi notebook ini.
- fp16 (bukan bf16) dipakai karena GPU T4 = arsitektur Turing, tidak punya tensor core bf16 yang baik seperti Ampere+.
- MODE TUNTAS berhenti di epoch penuh ATAU target perplexity ATAU budget habis - budget habis BUKAN berarti training selesai, tinggal Run all lagi untuk lanjut (checkpoint+step diakumulasi otomatis dari metadata checkpoint sendiri).
- Checkpoint hasil Colab di-push ke branch terpisah (`colab-gpu-training` secara default) supaya tidak bentrok dengan commit yang sedang berjalan di sandbox lain - perlu di-merge manual (PR) atau oleh sesi Rategoan berikutnya.

## Kebijakan Gudang Besar (Round 9)

- **Rak korpus = barang bersih.** Raw (dump Wikipedia, tgz+parquet jilid 2) tidak diarsipkan permanen - sel 10b (opsional) mengunggah hasil bersih (`idwiki-clean.jsonl`, `korpus-jilid2-clean.jsonl`) ke Release tag `korpus-jilid-1-clean` / `korpus-jilid-2-clean`. Setelah itu, raw asset di tag `Corpus` boleh dihapus manual.
- **Rak otak = checkpoint <100MB saja di git.** Checkpoint >100MB (batas keras GitHub, saat ini `massive200m` di 163,41MB) TIDAK di-git-add (`git_auto_commit()` sudah otomatis skip) - sel 13b (opsional) mempublikasikannya ke Release tag `checkpoint-<ukuran>`. TIDAK pakai Git LFS (kuota bandwidth gratis GitHub gampang habis).
- **Kenapa dua sel di atas opsional & harus dijalankan dari Colab, bukan sandbox:** sesi sandbox Claude Code Remote (tempat kode ini biasanya dikembangkan) diblokir dari operasi Release (`"Creating, editing, or deleting releases is not permitted for this session type"`) walau boleh membaca/mengunduh - jadi publikasi Release HARUS lewat sini (token Colab) atau mesin lokal. Detail: `raget-tools/CHECKPOINT-POLICY.md`.
- **Kenapa sel 3 punya fallback unduh langsung Wikimedia:** tag Release `Korpus` (dump Wikipedia mentah) sudah tidak ada lagi (kebijakan raw-tidak-disimpan) - sandbox tidak bisa mengunduh langsung dari `dumps.wikimedia.org` (diblokir kebijakan jaringan), tapi Colab bisa, jadi fallback-nya otomatis aktif di sini.
